# Dodge et al. 2021 §4.3, Arm B: sentiment of C4 paragraphs about "Jewish" and "Arab"

**What the paper did (A.7).** It picked "Jewish" and "Arab", the two groups with the most extreme model bias in
Table 7 that simple string matching could find. It then found every C4.EN paragraph containing " Jewish " or " Arab "
(spaces on both sides) and counted the positive and negative words from three lexicons (SocialSent-derived, VADER,
and a small one from the UnQover question words).
Result: SocialSent positivity 73.2% vs 65.7%, a gap of 7.5 points; VADER gap 2.0; UnQover gap 1.4.

| ID | Run | Status |
|---|---|---|
| B0 | One shard (00000) | done: reproduces (gap 7.3) |
| B1 | **Reproduction**: 12 seeded shards, Jewish and Arab | done: reproduces (gap 7.1 [5.6–8.7]; VADER 2.0) |
| B2 | Extension: all 15 Table 7 groups, one shard | done: corpus positivity does not track Table 7 (Spearman −0.05) |
| B3 | Extension: all 15 groups, the same 12 shards as B1 | **this notebook, optional** |
| B4 | Larger sample: a quarter of C4.EN (256 shards), so NYT / Al Jazeera get enough tokens | **this notebook, optional, free tier, multi-session** |

CPU only (free tier is enough). Each shard is a ~319 MB download from Hugging Face plus about 2 minutes of scanning;
`--workers` scans several shards in parallel. Runs resume, and a folder can only hold one set of target words.

**Robustness (audit 2026-10-23).** Besides the paper's numbers, every analysis now writes `robustness.csv` and
`fig_robustness.png`: the gap without "United Arab Emirates" paragraphs (22% of the "Arab" paragraphs), without
other geopolitical "Arab" phrases, without noisy SocialSent words, and without religious words. The UnQover lexicon
is v2 (tokenised like the corpus; v1 split hyphenated words and counted "well" / "mean" as sentiment).

## 1. Get the project

Either set `REPO_URL` to your GitHub repository, or leave it empty and upload a zip of the
`reproduction/f4_demographic_bias` folder when asked.

In [ ]:
import os, shutil, subprocess, zipfile
REPO_URL = ""   # e.g. "https://github.com/<you>/<repo>.git"
W = "/content/f4"
if not os.path.isdir(f"{W}/scripts"):
    if REPO_URL:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/content/repo"], check=True)
        shutil.copytree("/content/repo/reproduction/f4_demographic_bias", W)
    else:
        from google.colab import files
        up = files.upload()                      # choose the zip of reproduction/f4_demographic_bias
        name = next(iter(up))
        zipfile.ZipFile(name).extractall("/content/_upload")
        src = next(r for r, d, f in os.walk("/content/_upload") if "scripts" in d and "third_party" in d)
        shutil.copytree(src, W)
os.chdir(W)
os.makedirs("scripts", exist_ok=True)
print("working folder:", os.getcwd(), "| has UnQover word lists:", os.path.isdir("third_party/unqover"))

In [ ]:
!pip -q install huggingface_hub pandas matplotlib

## 2. Scripts (embedded from the repo, so the notebook runs the current code)

In [ ]:
%%writefile /content/f4/scripts/build_lexicons.py
"""Arm B of Dodge et al. 2021 §4.3 / A.7: build the three sentiment lexicons.

1. SocialSent-derived (the paper's primary lexicon). Hamilton et al. (2016) induced one lexicon per subreddit
   for 250 subreddits. A.7: "we aggregated these by averaging the sentiment scores for all words that appeared in
   at least 40 subreddit-specific lexicons ... manually filtered to remove any overtly ethnic terms, then took the
   top 250 most polarized words from each side". The Stanford download page is unreachable, so the 250 lexicons
   come from the copy bundled in shifterator (pinned commit below).
2. VADER (Hutto and Gilbert 2014): positive if mean score > 1.0, negative if < -1.0 (A.7).
3. UnQover-derived: the paper's "small manually-created" lexicon from the UnQover question words was not
   released. We rebuild an approximation from the UnQover attribute file: content words of the negative
   stereotypes are negative candidates, content words of their positive counterparts are positive candidates.
   Instead of a hand pass, a candidate is kept only if VADER or the SocialSent aggregate (|score| >= 0.5) gives
   it the same sign; this drops neutral words such as "house" or "because". Ethnic terms are removed.

Writes data/lexicons/{socialsent_250.tsv, vader.tsv, unqover.tsv} (word, polarity in {pos, neg}, score) plus
socialsent_aggregate.tsv (every word kept by the >=40 rule), socialsent_raw.json.gz (the 250 source lexicons)
and MANIFEST.json (sources, commits, counts).

Usage: python scripts/build_lexicons.py   (downloads the sources once, then reuses socialsent_raw.json.gz)
"""
import gzip, json, os, re, urllib.request
from collections import defaultdict

HERE = os.path.dirname(os.path.abspath(__file__))
ROOT = os.path.dirname(HERE)
OUT = os.path.join(ROOT, "data", "lexicons")

SHIFTERATOR_SHA = "8d74d74a94b5a046928f99ba076ff4fac74dd2c4"
VADER_SHA = "44fc044cd877310ee8278a0eadf34bcd50d41d06"
SS_TREE = f"https://api.github.com/repos/ryanjgallagher/shifterator/git/trees/{SHIFTERATOR_SHA}?recursive=1"
SS_RAW = f"https://raw.githubusercontent.com/ryanjgallagher/shifterator/{SHIFTERATOR_SHA}/"
VADER_URL = f"https://raw.githubusercontent.com/cjhutto/vaderSentiment/{VADER_SHA}/vaderSentiment/vader_lexicon.txt"
UNQOVER_ATTRS = os.path.join(ROOT, "third_party", "unqover", "word_lists", "activities", "biased_ethnicity")

MIN_LEXICONS = 40     # A.7
TOP_K = 250           # A.7
VADER_THRESHOLD = 1.0 # A.7

# Our manual filter of "overtly ethnic terms" (A.7 does not list theirs). Group names, demonyms and
# religion/nationality words; applied before taking the top 250 on each side. Reported as an assumption.
ETHNIC_TERMS = {
    "jewish", "jew", "jews", "arab", "arabs", "arabic", "muslim", "muslims", "islam", "islamic", "christian",
    "christians", "black", "blacks", "white", "whites", "african", "africans", "asian", "asians", "european",
    "europeans", "american", "americans", "mexican", "mexicans", "chinese", "japanese", "korean", "indian",
    "indians", "hispanic", "latino", "latina", "native", "natives", "caucasian", "israeli", "israelis",
    "palestinian", "palestinians", "russian", "russians", "german", "germans", "french", "british", "english",
    "irish", "italian", "italians", "canadian", "canadians", "australian", "australians", "swedish", "dutch",
    "polish", "greek", "turkish", "iranian", "iraqi", "syrian", "pakistani", "somali", "gypsy", "gypsies",
    "roma", "nigger", "niggers", "negro", "kike", "spic", "chink", "gook", "wetback", "raghead", "towelhead",
}

PUNCT = re.compile(r"[^\w\s]")   # the same as count_sentiment.py

STOPWORDS = {
    # function words after punctuation removal; "doing" ("doing nails"), "going" ("going to hijack") and "hard"
    # ("hard to understand"; also "hard drive", "hard work") carry no sentiment on their own
    "didnt", "wasnt", "werent", "doing", "going", "hard",
    "a", "an", "the", "of", "to", "and", "or", "in", "on", "at", "for", "with", "by", "from", "as", "their", "them",
    "they", "were", "was", "had", "have", "has", "been", "be", "is", "are", "very", "mostly", "always", "even",
    "didn't", "did", "not", "no", "how", "use", "other", "others", "others'", "about", "like", "looked", "seemed",
    "when", "into", "irrespective", "toward", "towards", "it", "its", "that", "this", "who", "all", "boot", "day",
    "room", "entering", "deal", "citizens", "street", "weekend", "work", "clothes", "wives", "daughters", "women",
    "children", "visitors", "tourists", "skin", "color", "accent", "accents", "teeth", "music", "dishes", "border",
    "computer", "cellphones", "pants", "money", "plane", "wilderness", "huts", "cartels", "alcohol",
}


def fetch(url):
    with urllib.request.urlopen(url, timeout=60) as r:
        return r.read().decode("utf-8")


def load_socialsent():
    cache = os.path.join(OUT, "socialsent_raw.json.gz")
    if os.path.exists(cache):
        with gzip.open(cache, "rt", encoding="utf-8") as f:
            return json.load(f)
    tree = json.loads(fetch(SS_TREE))["tree"]
    paths = sorted(t["path"] for t in tree if t["path"].startswith("shifterator/lexicons/SocialSent-Reddit/")
                   and t["path"].endswith(".tsv"))
    lexicons = {}
    for i, p in enumerate(paths):
        name = p.rsplit("SocialSent-Reddit_", 1)[1][:-4]
        lex = {}
        for line in fetch(SS_RAW + p).splitlines():
            if line.strip():
                w, s = line.split("\t")
                lex[w] = float(s)
        lexicons[name] = lex
        if (i + 1) % 50 == 0:
            print(f"  downloaded {i + 1}/{len(paths)} subreddit lexicons")
    with gzip.open(cache, "wt", encoding="utf-8") as f:
        json.dump(lexicons, f)
    return lexicons


def build_socialsent(lexicons):
    vals = defaultdict(list)
    for lex in lexicons.values():
        for w, s in lex.items():
            vals[w].append(s)
    agg = {w: sum(v) / len(v) for w, v in vals.items() if len(v) >= MIN_LEXICONS}
    counts = {w: len(vals[w]) for w in agg}
    kept = {w: s for w, s in agg.items() if w not in ETHNIC_TERMS}
    removed = sorted(w for w in agg if w in ETHNIC_TERMS)
    pos = sorted((w for w in kept if kept[w] > 0), key=lambda w: -kept[w])[:TOP_K]
    neg = sorted((w for w in kept if kept[w] < 0), key=lambda w: kept[w])[:TOP_K]
    return agg, counts, removed, [(w, "pos", kept[w]) for w in pos] + [(w, "neg", kept[w]) for w in neg]


def build_vader():
    rows, scores = [], {}
    for line in fetch(VADER_URL).splitlines():
        parts = line.split("\t")
        if len(parts) < 2:
            continue
        w, s = parts[0], float(parts[1])
        scores[w] = s
        if s > VADER_THRESHOLD:
            rows.append((w, "pos", s))
        elif s < -VADER_THRESHOLD:
            rows.append((w, "neg", s))
    return rows, scores


def build_unqover(vader_scores, ss_agg):
    pos, neg = set(), set()
    for line in open(UNQOVER_ATTRS, encoding="utf-8"):
        line = line.strip()
        if not line.startswith("[p_act") or "|" not in line:
            continue
        phrase = line.split("|", 1)[1]
        # tokenise exactly as count_sentiment.py tokenises C4 (whitespace split, punctuation removed, lowercased),
        # so "well-behaved" stays one word ("wellbehaved"). v1 split on hyphens and added "well" and "mean" as
        # sentiment words, which then dominated the counts (kept as unqover_v1_hyphen_split.tsv).
        words = {w for w in (PUNCT.sub("", t).lower() for t in phrase.split()) if w not in STOPWORDS and len(w) > 2}
        # slot tags: [p_act], [p_act:neg], [p_act-single], [p_act-single:neg]; ":neg" marks the positive counterpart
        (pos if line.split("]", 1)[0].endswith(":neg") else neg).update(words)
    both = pos & neg

    def agrees(w, sign):
        refs = [vader_scores.get(w), ss_agg.get(w) if abs(ss_agg.get(w, 0)) >= 0.5 else None]
        return w not in ETHNIC_TERMS and any(r is not None and r * sign > 0 for r in refs)

    return ([(w, "pos", 1.0) for w in sorted(pos - both) if agrees(w, 1)] +
            [(w, "neg", -1.0) for w in sorted(neg - both) if agrees(w, -1)])


def write(name, rows):
    with open(os.path.join(OUT, name), "w", encoding="utf-8") as f:
        f.write("word\tpolarity\tscore\n")
        for w, p, s in rows:
            f.write(f"{w}\t{p}\t{s:.4f}\n")


def main():
    os.makedirs(OUT, exist_ok=True)
    lexicons = load_socialsent()
    agg, counts, removed, ss = build_socialsent(lexicons)
    with open(os.path.join(OUT, "socialsent_aggregate.tsv"), "w", encoding="utf-8") as f:
        f.write("word\tmean_score\tn_lexicons\n")
        for w in sorted(agg, key=lambda w: -agg[w]):
            f.write(f"{w}\t{agg[w]:.4f}\t{counts[w]}\n")
    write("socialsent_250.tsv", ss)
    vader, vader_scores = build_vader()
    write("vader.tsv", vader)
    unq = build_unqover(vader_scores, agg)
    write("unqover.tsv", unq)
    n = lambda rows, p: sum(1 for r in rows if r[1] == p)
    manifest = {
        "socialsent": {"source": f"shifterator@{SHIFTERATOR_SHA} SocialSent-Reddit", "subreddit_lexicons": len(lexicons),
                       "words_in_>=40_lexicons": len(agg), "ethnic_terms_removed": removed,
                       "pos": n(ss, "pos"), "neg": n(ss, "neg")},
        "vader": {"source": f"vaderSentiment@{VADER_SHA}", "threshold": VADER_THRESHOLD,
                  "pos": n(vader, "pos"), "neg": n(vader, "neg")},
        "unqover": {"source": "third_party/unqover word_lists/activities/biased_ethnicity (our reconstruction, "
                              "sign-checked against VADER / SocialSent aggregate)",
                    "pos": n(unq, "pos"), "neg": n(unq, "neg")},
    }
    with open(os.path.join(OUT, "MANIFEST.json"), "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=1)
    print(json.dumps(manifest, indent=1))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/f4/scripts/scan_c4_paragraphs.py
"""Arm B, step 1: extract C4.EN paragraphs that mention a target ethnicity (Dodge et al. 2021, A.7).

A.7: "we first found all paragraphs containing these words, where the word was surrounded by spaces (for easy
searching using fgrep ...)". So a paragraph (a newline-separated line of a C4 document) matches "Jewish" if it
contains " Jewish ", case-sensitive, as fgrep does by default. The same matching was used in pilots/shard_check.py.

Shards come from huggingface.co/datasets/allenai/c4 (en/c4-train.XXXXX-of-01024.json.gz, ~319 MB each, 1/1024
of C4.EN). By default 12 shard indices are drawn with a fixed seed. Each shard is downloaded, scanned, and
deleted unless --keep_shards is given. Finished shards are skipped, so the scan resumes.

The paper's corpus study covers only "Jewish" and "Arab" (A.7: "the ethnicities with the most extreme biases that
were easy to find in C4.EN with simple scripts"); that is the default. --table7_groups is our extension: all 15
UnQover groups, each matched by its spelling variants (e.g. "Middle-Eastern" or "Middle Eastern"). Its known
limits, which the paper names: "African" also matches "African American", and "Black" / "White" have very common
non-ethnic senses ("Black Friday", "White House"); "Asian", "European", "Native American" etc. are similar.

Writes <out_dir>/paragraphs_<shard>.jsonl.gz (one line per matching paragraph: shard, url, targets, text) and
<out_dir>/stats_<shard>.json (documents, paragraphs, matches and words per target, and the targets scanned).
A finished shard is skipped only if it was scanned with the same targets; otherwise the script stops, so one
folder never mixes target sets.

Free tier: downloads dominate (~319 MB per shard), so --workers 3-4 scans shards in parallel even on 2-4 vCPUs.
--range START END picks a contiguous block of shards, so a large scan can be split across sessions into the
same --out_dir (e.g. on Google Drive); each session resumes where the last one stopped.

Usage:
    python scripts/scan_c4_paragraphs.py --out_dir results/B1_c4_paragraphs              # 12 seeded shards
    python scripts/scan_c4_paragraphs.py --shards 0 1 2 --out_dir ...                    # explicit shards
    python scripts/scan_c4_paragraphs.py --files /path/c4-train.00000-of-01024.json.gz   # local files
    python scripts/scan_c4_paragraphs.py --table7_groups --out_dir ...                   # extension: all 15 groups
    python scripts/scan_c4_paragraphs.py --range 0 256 --workers 4 --out_dir ...         # a quarter of C4.EN
"""
import argparse, gzip, json, os, random, re, time
from concurrent.futures import ProcessPoolExecutor, as_completed

TARGETS_DEFAULT = ["Jewish", "Arab"]
# extension: the 15 UnQover / Table 7 groups -> the strings matched (with a space on each side, case-sensitive)
TABLE7_GROUPS = {
    "Jewish": ["Jewish"], "Arab": ["Arab"], "Asian": ["Asian"], "Caucasian": ["Caucasian"],
    "European": ["European"], "White": ["White"], "Alaskan": ["Alaskan"], "Hispanic": ["Hispanic"],
    "Native American": ["Native American"], "South-American": ["South-American", "South American"],
    "African-American": ["African-American", "African American"], "Latino": ["Latino"],
    "Middle-Eastern": ["Middle-Eastern", "Middle Eastern"], "Black": ["Black"], "African": ["African"],
}
N_SHARDS = 1024


def shard_name(i):
    return f"c4-train.{i:05d}-of-{N_SHARDS:05d}"


def as_targets(t):
    """Normalise stored targets: older shards.json files hold a list of words, newer ones {name: [strings]}."""
    return {x: [x] for x in t} if isinstance(t, list) else t


def scan_file(path, shard, targets, out_dir):
    """targets: {name: [strings]}; a paragraph matches a name if it contains any string with spaces around it."""
    pats = {t: [f" {v} " for v in variants] for t, variants in targets.items()}
    stats = {"shard": shard, "targets": targets, "documents": 0, "paragraphs": 0,
             "matched_paragraphs": {t: 0 for t in targets}, "words_in_matched_paragraphs": {t: 0 for t in targets}}
    tmp = os.path.join(out_dir, f"paragraphs_{shard}.jsonl.gz.tmp")
    with gzip.open(path, "rt", encoding="utf-8") as f, gzip.open(tmp, "wt", encoding="utf-8") as g:
        for line in f:
            d = json.loads(line)
            stats["documents"] += 1
            for p in d["text"].split("\n"):
                stats["paragraphs"] += 1
                hit = [t for t, ps in pats.items() if any(x in p for x in ps)]
                if hit:
                    n = len(p.split())
                    for t in hit:
                        stats["matched_paragraphs"][t] += 1
                        stats["words_in_matched_paragraphs"][t] += n
                    g.write(json.dumps({"shard": shard, "url": d["url"], "targets": hit, "text": p},
                                       ensure_ascii=False) + "\n")
    os.replace(tmp, os.path.join(out_dir, f"paragraphs_{shard}.jsonl.gz"))
    with open(os.path.join(out_dir, f"stats_{shard}.json"), "w") as f:
        json.dump(stats, f, indent=1)
    return stats


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--out_dir", required=True)
    ap.add_argument("--shards", type=int, nargs="*", help="shard indices 0-1023 (default: --n seeded shards)")
    ap.add_argument("--n", type=int, default=12)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--files", nargs="*", help="local shard files instead of downloading")
    ap.add_argument("--targets", nargs="+", default=TARGETS_DEFAULT, help="words matched exactly (paper: Jewish Arab)")
    ap.add_argument("--table7_groups", action="store_true", help="extension: all 15 Table 7 groups with variants")
    ap.add_argument("--keep_shards", action="store_true")
    ap.add_argument("--cache_dir", default=None, help="where downloaded shards go (default: HF cache)")
    ap.add_argument("--range", type=int, nargs=2, metavar=("START", "END"), help="shards START..END-1")
    ap.add_argument("--workers", type=int, default=1, help="shards scanned in parallel (downloads dominate)")
    args = ap.parse_args()
    os.makedirs(args.out_dir, exist_ok=True)
    targets = TABLE7_GROUPS if args.table7_groups else {t: [t] for t in args.targets}

    if args.files:
        jobs = [(re.search(r"c4-train\.\d{5}-of-\d{5}", os.path.basename(p)).group(0), p) for p in args.files]
    else:
        if args.shards:
            idx = args.shards
        elif args.range:
            idx = list(range(max(0, args.range[0]), min(N_SHARDS, args.range[1])))
        else:
            idx = sorted(random.Random(args.seed).sample(range(N_SHARDS), args.n))
        jobs = [(shard_name(i), None) for i in idx]

    # one folder = one target set; shards.json accumulates the shards of every session
    meta_path = os.path.join(args.out_dir, "shards.json")
    meta = json.load(open(meta_path)) if os.path.exists(meta_path) else {"shards": [], "targets": targets}
    if as_targets(meta["targets"]) != targets:
        raise SystemExit(f"{args.out_dir} was scanned with other targets {list(meta['targets'])}; use another --out_dir")
    meta["shards"] = sorted(set(meta["shards"]) | {s for s, _ in jobs})
    with open(meta_path, "w") as f:
        json.dump(meta, f, indent=1)

    todo = []
    for shard, path in jobs:
        sp = os.path.join(args.out_dir, f"stats_{shard}.json")
        if os.path.exists(sp):
            old = json.load(open(sp)).get("targets")      # absent in stats written before 2026-10-23
            if old is not None and as_targets(old) != targets:
                raise SystemExit(f"{sp} was scanned with other targets; use another --out_dir")
            continue
        todo.append((shard, path))
    print(f"{len(jobs)} shards requested, {len(jobs) - len(todo)} done already, {len(todo)} to scan", flush=True)

    t_all = time.time()
    if args.workers <= 1:
        for shard, path in todo:
            print(process(shard, path, targets, args), flush=True)
    else:
        with ProcessPoolExecutor(args.workers) as ex:
            futs = [ex.submit(process, shard, path, targets, args) for shard, path in todo]
            for i, fu in enumerate(as_completed(futs), 1):
                print(f"[{i}/{len(todo)}, {(time.time() - t_all) / 60:.0f} min] {fu.result()}", flush=True)


def process(shard, path, targets, args):
    t0 = time.time()
    downloaded = False
    if path is None:
        from huggingface_hub import hf_hub_download
        path = hf_hub_download("allenai/c4", f"en/{shard}.json.gz", repo_type="dataset", cache_dir=args.cache_dir)
        downloaded = True
    s = scan_file(path, shard, targets, args.out_dir)
    if downloaded and not args.keep_shards:
        os.remove(os.path.realpath(path))
    return f"{shard}: {s['documents']} docs, matches {s['matched_paragraphs']}, {time.time() - t0:.0f}s"


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/f4/scripts/count_sentiment.py
"""Arm B, step 2: sentiment of C4 paragraphs that mention an ethnicity (Dodge et al. 2021, §4.3 / A.7).

A.7: paragraphs are "tokenized ... by whitespace, removed all punctuation"; for each lexicon "we counted the number
of positive and negative word occurrences in paragraphs containing the ethnicity word, then found the proportion of
these occurrences that had positive sentiment." A paragraph mentioning two groups counts for both.
Tokens are lowercased before lookup (the lexicons are lowercase; the paper does not say) - our assumption.

The paper's corpus study is "Jewish" vs "Arab"; those numbers (positivity, gap, NYT / Al Jazeera, scale) are the
reproduction. If the paragraphs were scanned with --table7_groups (our extension), every group also gets a
positivity per lexicon, and the SocialSent positivity is compared with the paper's Table 7 (model) positivity.

CIs: percentile bootstrap resampling web domains (URL host), B=1000, because paragraphs from one site are not
independent. The NYT / Al Jazeera subsets use the paper's URL prefixes.

Usage:
    python scripts/count_sentiment.py --paragraphs results/B1_c4_12_shards --out_dir results/B1_c4_12_shards/analysis

Writes positivity.csv, gaps.csv, domains.csv, scale.csv, top_tokens.csv, summary.md, fig_positivity.png and
fig_gap.png (Jewish vs Arab, as in the paper); robustness.csv and fig_robustness.png (the gap with non-ethnic
"Arab" text such as "United Arab Emirates" removed, without noisy SocialSent words, without religious words);
with more groups also fig_positivity_all_groups.png and fig_corpus_vs_table7.png.
"""
import argparse, csv, glob, gzip, json, os, re
from collections import Counter, defaultdict
from urllib.parse import urlparse

import numpy as np
import pandas as pd

HERE = os.path.dirname(os.path.abspath(__file__))
LEX_DIR = os.path.join(os.path.dirname(HERE), "data", "lexicons")
LEXICONS = {"SocialSent": "socialsent_250.tsv", "VADER": "vader.tsv", "UnQover": "unqover.tsv"}
PAIR = ["Jewish", "Arab"]   # the paper's comparison
DOMAINS = {"New York Times": "https://www.nytimes.com", "Al Jazeera": "https://www.aljazeera.com"}

# Dodge et al. 2021, §4.3 and A.7 (full C4.EN)
PAPER = {
    "positivity": {("SocialSent", "Jewish"): 73.2, ("SocialSent", "Arab"): 65.7},
    "gap": {"SocialSent": 7.5, "VADER": 2.0, "UnQover": 1.4},
    "sentiment_tokens": {"Jewish": 3.4e6, "Arab": 1.2e6},
    "words_in_paragraphs": {"Jewish": 249.8e6, "Arab": 134.8e6},
    "domain_positivity": {("New York Times", "Jewish"): 74.0, ("New York Times", "Arab"): 69.5,
                          ("Al Jazeera", "Jewish"): 42.5, ("Al Jazeera", "Arab"): 42.8},
}
# Dodge et al. 2021, Table 7 (UnifiedQA positivity), for the extension's corpus-vs-model comparison
TABLE7 = {"Jewish": 67.1, "Asian": 60.6, "Caucasian": 60.5, "European": 60.5, "White": 56.5,
          "Alaskan": 55.9, "Hispanic": 50.8, "Native American": 50.6, "South-American": 44.4,
          "African-American": 44.3, "Latino": 43.1, "Middle-Eastern": 42.6, "Black": 39.3,
          "Arab": 37.0, "African": 36.6}
PUNCT = re.compile(r"[^\w\s]")

# Robustness variants (audit 2026-10-23, C1 / C3). "as in paper" is the reproduction; each other variant changes
# one thing, to show how much of the Jewish-Arab gap comes from non-ethnic text or from noisy lexicon words.
ARAB_GEOPOLITICAL = ["United Arab Emirates", "Arab Emirates", "Arab League", "Arab Spring", "Arab world",
                     "Arab countries", "Arab states", "Arab nations"]
# SocialSent top-250 words that are Reddit artefacts rather than sentiment ("p" is ":p"; in C4 it is "p." page refs)
SS_NOISE = {"man", "mr", "series", "due", "saying", "calling", "thats", "damnit"}
RELIGION = {"god", "gods", "holy", "lord", "holiday", "holidays", "celebrate", "celebration", "christmas", "faith",
            "church", "prayer", "pray", "blessed", "bless", "blessing", "heaven", "jesus", "christ", "spiritual",
            "worship", "sacred", "divine"}
VARIANTS = {
    "as in paper": {},
    "Arab: no 'United Arab Emirates' paragraphs": {"drop_par": ["United Arab Emirates"]},
    "Arab: no geopolitical 'Arab' phrases": {"drop_par": ARAB_GEOPOLITICAL},
    "no 1-letter tokens or SocialSent noise words": {"drop_tok": SS_NOISE, "drop_short": True},
    "no religious words": {"drop_tok": RELIGION},
}

# reference palette (dataviz skill, validated): Jewish = blue, Arab = orange; paper = hollow ink diamond
C = {"Jewish": "#2a78d6", "Arab": "#eb6834"}
C_LEX = {"SocialSent": "#2a78d6", "VADER": "#1baf7a", "UnQover": "#6250d6"}
INK, INK2, MUTED, GRID, SURFACE = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#fcfcfb"


def load_lexicons():
    lex = {}
    for name, fn in LEXICONS.items():
        with open(os.path.join(LEX_DIR, fn), encoding="utf-8") as f:
            lex[name] = {r["word"]: r["polarity"] for r in csv.DictReader(f, delimiter="\t")}
    return lex


def tokens(text):
    return [t for t in (PUNCT.sub("", w).lower() for w in text.split()) if t]


def scanned_targets(paragraph_dir, files):
    """Target names in scan order: from shards.json if present, else from the paragraphs themselves."""
    path = os.path.join(paragraph_dir, "shards.json")
    if os.path.exists(path):
        t = json.load(open(path))["targets"]
        names = list(t) if isinstance(t, dict) else list(t)
    else:
        names = []
        for p in files:
            for line in gzip.open(p, "rt", encoding="utf-8"):
                names += [x for x in json.loads(line)["targets"] if x not in names]
    return [x for x in PAIR if x in names] + [x for x in names if x not in PAIR]


def count(paragraph_files, lex, targets):
    """Per (domain, target, lexicon): positive and negative token counts; plus scale and token counters.

    Returns per_domain for the paper's setting and `robust`: {variant: per_domain} for every VARIANTS entry.
    """
    T, L = len(targets), len(LEXICONS)
    robust = {v: defaultdict(lambda: np.zeros((T, L, 2))) for v in VARIANTS}
    kept = Counter()                       # (variant, target) -> paragraphs counted
    per_domain = robust["as in paper"]
    subset = defaultdict(lambda: np.zeros((T, L, 2)))
    top = {(t, l): Counter() for t in targets for l in LEXICONS}
    scale = Counter()
    shards = set()
    for path in paragraph_files:
        with gzip.open(path, "rt", encoding="utf-8") as f:
            for line in f:
                r = json.loads(line)
                shards.add(r["shard"])
                toks = tokens(r["text"])
                host = urlparse(r["url"]).netloc
                hits = {}
                for lname, words in lex.items():
                    hits[lname] = [(w, words[w]) for w in toks if w in words]
                for t in r["targets"]:
                    if t not in targets:
                        continue
                    ti = targets.index(t)
                    scale[f"{t} paragraphs"] += 1
                    scale[f"{t} words"] += len(toks)
                    for li, lname in enumerate(lex):
                        pos = sum(1 for _, p in hits[lname] if p == "pos")
                        neg = len(hits[lname]) - pos
                        top[(t, lname)].update(w for w, _ in hits[lname])
                        for dname, prefix in DOMAINS.items():
                            if r["url"].startswith(prefix):
                                subset[dname][ti, li] += (pos, neg)
                    for v, spec in VARIANTS.items():
                        # paragraph filters apply to "Arab" only: they remove non-ethnic senses of that word
                        if t == "Arab" and any(x in r["text"] for x in spec.get("drop_par", ())):
                            continue
                        kept[(v, t)] += 1
                        drop, short = spec.get("drop_tok", set()), spec.get("drop_short", False)
                        for li, lname in enumerate(lex):
                            h = [p for w, p in hits[lname] if w not in drop and not (short and len(w) == 1)]
                            pos = sum(1 for p in h if p == "pos")
                            robust[v][host][ti, li] += (pos, len(h) - pos)
    return per_domain, robust, kept, subset, top, scale, sorted(shards)


def boot(per_domain, B, seed, batch=50):
    """Domain bootstrap. Replicates are drawn in batches so a full-corpus run (~10^5 domains) fits in free Colab RAM."""
    hosts = list(per_domain)
    X = np.stack([per_domain[h] for h in hosts])                  # domains x targets x lexicons x 2
    tot = X.sum(0)
    with np.errstate(invalid="ignore", divide="ignore"):
        point = 100 * tot[..., 0] / tot.sum(-1)
    rng = np.random.default_rng(seed)
    p = np.full(len(hosts), 1 / len(hosts))
    pb = []
    for b0 in range(0, B, batch):
        w = rng.multinomial(len(hosts), p, size=min(batch, B - b0))  # batch x domains
        S = np.einsum("bd,dtlk->btlk", w, X)
        with np.errstate(invalid="ignore", divide="ignore"):
            pb.append(100 * S[..., 0] / S.sum(-1))                  # batch x targets x lexicons
    return tot, point, np.concatenate(pb), len(hosts)


def robustness(robust, kept, targets, B, seed):
    """Jewish-Arab gap per lexicon under each variant, with domain-bootstrap CIs."""
    a, b = targets.index("Jewish"), targets.index("Arab")
    rows = []
    for v in VARIANTS:
        _, point, pb, _ = boot(robust[v], B, seed)
        for li, l in enumerate(LEXICONS):
            g = pb[:, a, li] - pb[:, b, li]
            lo, hi = np.nanpercentile(g, [2.5, 97.5])
            rows.append({"variant": v, "lexicon": l, "Jewish": point[a, li], "Arab": point[b, li],
                         "gap": point[a, li] - point[b, li], "ci_lo": lo, "ci_hi": hi,
                         "Jewish_paragraphs": kept[(v, "Jewish")], "Arab_paragraphs": kept[(v, "Arab")],
                         "paper_gap": PAPER["gap"][l]})
    return pd.DataFrame(rows)


def fig_robustness(rob, out, dpi, title, plt):
    """Gap per lexicon (panels) under each variant (rows), 95% CIs; the paper's gap as a dashed line."""
    vs = list(VARIANTS)[::-1]
    y = np.arange(len(vs))
    fig, axes = plt.subplots(1, len(LEXICONS), figsize=(4.4 * len(LEXICONS), 0.55 * len(vs) + 1.9),
                             sharey=True, facecolor=SURFACE)
    lim = np.nanmax(np.abs(rob[["ci_lo", "ci_hi"]].to_numpy())) + 1.5
    for ax, l in zip(axes, LEXICONS):
        style(ax)
        d = rob[rob["lexicon"] == l].set_index("variant").reindex(vs)
        v = d["gap"].to_numpy()
        ax.errorbar(v, y, xerr=[v - d["ci_lo"].to_numpy(), d["ci_hi"].to_numpy() - v], fmt="o", color=C_LEX[l],
                    ms=7, mec=SURFACE, mew=1.3, ecolor=C_LEX[l], elinewidth=1.4, capsize=3, zorder=3)
        for yi, vi, hi in zip(y, v, d["ci_hi"].to_numpy()):
            ax.text(hi + 0.4, yi, f"{vi:+.1f}", va="center", fontsize=8, color=INK2)
        ax.axvline(0, color=INK2, linewidth=1)
        ax.axvline(PAPER["gap"][l], color=INK, linewidth=1, linestyle=(0, (3, 3)))
        ax.set_xlim(-lim, lim)
        ax.set_title(f"{l} (dashed = paper, {PAPER['gap'][l]:+.1f})", color=INK, loc="left", fontsize=10)
        ax.set_xlabel("Jewish minus Arab positivity (points)", color=INK2)
    axes[0].set_yticks(y, vs, color=INK, fontsize=8.5)
    fig.suptitle(f"{title}\nEach row changes one thing from the paper's setting; 95% CIs from a domain bootstrap",
                 color=INK, x=0.01, ha="left", fontsize=11)
    fig.tight_layout()
    fig.savefig(os.path.join(out, "fig_robustness.png"), dpi=dpi, facecolor=SURFACE, bbox_inches="tight")
    plt.close(fig)


def md_table(df):
    lines = ["| " + " | ".join(df.columns) + " |", "|" + "---|" * len(df.columns)]
    lines += ["| " + " | ".join(str(v) for v in r) + " |" for r in df.itertuples(index=False)]
    return "\n".join(lines) + "\n"


def style(ax):
    ax.set_facecolor(SURFACE)
    for s in ("top", "right", "left"):
        ax.spines[s].set_visible(False)
    ax.spines["bottom"].set_color(GRID)
    ax.tick_params(colors=INK2, length=0)
    ax.grid(axis="x", color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)


def fig_pair(pos_df, gap_df, out, dpi, title, plt):
    """The paper's comparison: Jewish vs Arab positivity per lexicon, and the gap."""
    from matplotlib.lines import Line2D
    from matplotlib.patches import Patch
    lexs = list(LEXICONS)[::-1]
    y = np.arange(len(lexs))
    fig, ax = plt.subplots(figsize=(8, 4.2), facecolor=SURFACE)
    style(ax)
    for t, dy in (("Jewish", 0.19), ("Arab", -0.19)):
        d = pos_df[pos_df["target"] == t].set_index("lexicon").reindex(lexs)
        v = d["positivity"].values
        ax.barh(y + dy, v, height=0.36, color=C[t], edgecolor=SURFACE, linewidth=1, zorder=2)
        ax.errorbar(v, y + dy, xerr=[v - d["ci_lo"].values, d["ci_hi"].values - v], fmt="none", ecolor=INK2,
                    elinewidth=1.2, capsize=3, zorder=3)
        for yi, vi, hi in zip(y + dy, v, d["ci_hi"].values):
            ax.text(hi + 0.8, yi, f"{vi:.1f}", va="center", fontsize=8, color=INK2)
        for li, l in enumerate(lexs):
            pv = PAPER["positivity"].get((l, t))
            if pv is not None:
                ax.scatter([pv], [li + dy], s=46, marker="D", facecolor=SURFACE, edgecolor=INK, linewidth=1.4, zorder=4)
    ax.set_yticks(y, lexs, color=INK)
    ax.set_xlim(0, 105)
    ax.set_xlabel("Positivity (%) = positive / (positive + negative) lexicon tokens in matching paragraphs", color=INK2)
    ax.legend(handles=[Patch(color=C["Jewish"], label="Jewish"), Patch(color=C["Arab"], label="Arab"),
                       Line2D([], [], color=INK2, marker="|", markersize=8, label="95% CI (domain bootstrap)"),
                       Line2D([], [], marker="D", color=INK, markerfacecolor=SURFACE, linestyle="none",
                              label="Dodge et al. 2021 (SocialSent only)")],
              loc="lower left", bbox_to_anchor=(0, 1.0), ncol=2, frameon=False, labelcolor=INK2, fontsize=8.5)
    ax.set_title(title, color=INK, loc="left", fontsize=12, pad=44)
    fig.tight_layout()
    fig.savefig(os.path.join(out, "fig_positivity.png"), dpi=dpi, facecolor=SURFACE, bbox_inches="tight")
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(8, 3.4), facecolor=SURFACE)
    style(ax)
    d = gap_df.set_index("lexicon").reindex(lexs)
    v = d["gap"].values
    ax.barh(y, v, height=0.55, color=["#2a78d6" if x >= 0 else "#e34948" for x in v], edgecolor=SURFACE, zorder=2)
    ax.errorbar(v, y, xerr=[v - d["ci_lo"].values, d["ci_hi"].values - v], fmt="none", ecolor=INK2, elinewidth=1.2,
                capsize=3, zorder=3)
    ax.scatter(d["paper_gap"].values, y, s=46, marker="D", facecolor=SURFACE, edgecolor=INK, linewidth=1.4, zorder=4)
    ax.axvline(0, color=INK2, linewidth=1)
    for yi, vi, hi in zip(y, v, d["ci_hi"].values):
        ax.text(max(hi, vi) + 0.3, yi, f"{vi:+.1f}", va="center", fontsize=8, color=INK2)
    ax.set_yticks(y, lexs, color=INK)
    ax.set_xlabel("Positivity gap, Jewish minus Arab (percentage points)", color=INK2)
    ax.legend(handles=[Patch(color="#2a78d6", label="ours: Jewish more positive"),
                       Patch(color="#e34948", label="ours: Arab more positive"),
                       Line2D([], [], color=INK2, marker="|", markersize=8, label="95% CI"),
                       Line2D([], [], marker="D", color=INK, markerfacecolor=SURFACE, linestyle="none",
                              label="Dodge et al. 2021")],
              loc="lower left", bbox_to_anchor=(0, 1.0), ncol=2, frameon=False, labelcolor=INK2, fontsize=8.5)
    ax.set_title("Jewish-Arab positivity gap by lexicon", color=INK, loc="left", fontsize=12, pad=44)
    lim = np.nanmax(np.abs(np.r_[d["ci_lo"], d["ci_hi"], d["paper_gap"]])) + 2
    ax.set_xlim(-lim, lim)
    fig.tight_layout()
    fig.savefig(os.path.join(out, "fig_gap.png"), dpi=dpi, facecolor=SURFACE, bbox_inches="tight")
    plt.close(fig)


def fig_all_groups(pos_df, out, dpi, title, plt):
    """Extension: positivity of every scanned group, one panel per lexicon, sorted by SocialSent."""
    order = (pos_df[pos_df["lexicon"] == "SocialSent"].sort_values("positivity")["target"].tolist())
    y = np.arange(len(order))
    fig, axes = plt.subplots(1, len(LEXICONS), figsize=(4.6 * len(LEXICONS), 0.42 * len(order) + 1.8),
                             sharey=True, facecolor=SURFACE)
    for ax, l in zip(axes, LEXICONS):
        style(ax)
        d = pos_df[pos_df["lexicon"] == l].set_index("target").reindex(order)
        v = d["positivity"].values
        ax.errorbar(v, y, xerr=[v - d["ci_lo"].values, d["ci_hi"].values - v], fmt="o", color=C_LEX[l], ms=6,
                    mec=SURFACE, mew=1.2, ecolor=C_LEX[l], elinewidth=1.3, capsize=2.5, zorder=3)
        pooled = 100 * d["pos"].sum() / d["sentiment_tokens"].sum()
        ax.axvline(pooled, color=MUTED, linewidth=1, linestyle=(0, (3, 3)))
        ax.set_title(f"{l} (dashed = all groups pooled)", color=INK, loc="left", fontsize=10)
        ax.set_xlabel("Positivity (%)", color=INK2)
    axes[0].set_yticks(y, [f"{t}  (n={int(n):,})" for t, n in
                           zip(order, pos_df[pos_df["lexicon"] == "SocialSent"].set_index("target").reindex(order)["sentiment_tokens"])],
                       color=INK)
    fig.suptitle(f"{title}\nn = SocialSent sentiment tokens; 95% CIs from a domain bootstrap. Black, White, African "
                 "and similar words also match non-ethnic uses", color=INK, x=0.01, ha="left", fontsize=11)
    fig.tight_layout()
    fig.savefig(os.path.join(out, "fig_positivity_all_groups.png"), dpi=dpi, facecolor=SURFACE, bbox_inches="tight")
    plt.close(fig)


def fig_corpus_vs_table7(pos_df, out, dpi, plt):
    """Extension: does the corpus explain the model? SocialSent corpus positivity vs the paper's Table 7."""
    d = pos_df[(pos_df["lexicon"] == "SocialSent") & pos_df["target"].isin(TABLE7)].copy()
    d["table7"] = d["target"].map(TABLE7)
    rho = d["positivity"].rank().corr(d["table7"].rank())
    fig, ax = plt.subplots(figsize=(7, 6), facecolor=SURFACE)
    style(ax)
    ax.grid(axis="y", color=GRID, linewidth=0.8)
    v = d["positivity"].values
    ax.errorbar(d["table7"], v, yerr=[v - d["ci_lo"].values, d["ci_hi"].values - v], fmt="o", color=C_LEX["SocialSent"],
                ms=7, mec=SURFACE, mew=1.5, ecolor=C_LEX["SocialSent"], elinewidth=1.3, capsize=3, zorder=3)
    for t, x, yv in zip(d["target"], d["table7"], v):
        key = t in PAIR
        ax.annotate(t, (x, yv), xytext=(6, 3), textcoords="offset points", fontsize=8.5 if key else 7.5,
                    color=INK if key else INK2, fontweight="bold" if key else "normal")
    ax.set_xlabel("Model: Dodge et al. 2021 Table 7 positivity (UnifiedQA, %)", color=INK2)
    ax.set_ylabel("Corpus: SocialSent positivity in C4 paragraphs (%), 95% CI", color=INK2)
    ax.set_title(f"Extension: corpus sentiment vs model bias, {len(d)} groups (Spearman {rho:.2f})\n"
                 "Word-sense noise (Black, White, African, ...) affects the corpus axis", color=INK, loc="left", fontsize=11)
    fig.tight_layout()
    fig.savefig(os.path.join(out, "fig_corpus_vs_table7.png"), dpi=dpi, facecolor=SURFACE, bbox_inches="tight")
    plt.close(fig)
    return rho


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--paragraphs", required=True, help="folder with paragraphs_*.jsonl.gz")
    ap.add_argument("--out_dir", required=True)
    ap.add_argument("--B", type=int, default=1000)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--dpi", type=int, default=300)
    args = ap.parse_args()
    os.makedirs(args.out_dir, exist_ok=True)
    files = sorted(glob.glob(os.path.join(args.paragraphs, "paragraphs_*.jsonl.gz")))
    if not files:
        raise SystemExit(f"no paragraphs_*.jsonl.gz in {args.paragraphs}")
    targets = scanned_targets(args.paragraphs, files)
    lex = load_lexicons()
    per_domain, robust, kept, subset, top, scale, shards = count(files, lex, targets)
    n = len(shards)

    tot, point, pb, n_hosts = boot(per_domain, args.B, args.seed)
    rows = []
    for ti, t in enumerate(targets):
        for li, l in enumerate(LEXICONS):
            lo, hi = np.nanpercentile(pb[:, ti, li], [2.5, 97.5])
            rows.append({"target": t, "lexicon": l, "pos": int(tot[ti, li, 0]), "neg": int(tot[ti, li, 1]),
                         "sentiment_tokens": int(tot[ti, li].sum()), "positivity": point[ti, li],
                         "ci_lo": lo, "ci_hi": hi, "paper": PAPER["positivity"].get((l, t))})
    pos_df = pd.DataFrame(rows)

    has_pair = all(t in targets for t in PAIR)
    gap_df = pd.DataFrame()
    if has_pair:
        a, b = targets.index("Jewish"), targets.index("Arab")
        gaps = []
        for li, l in enumerate(LEXICONS):
            g = pb[:, a, li] - pb[:, b, li]
            lo, hi = np.nanpercentile(g, [2.5, 97.5])
            gaps.append({"lexicon": l, "gap": point[a, li] - point[b, li], "ci_lo": lo, "ci_hi": hi,
                         "share_of_bootstrap_gap_>0": float(np.mean(g > 0)), "paper_gap": PAPER["gap"][l]})
        gap_df = pd.DataFrame(gaps)
        rob_df = robustness(robust, kept, targets, args.B, args.seed)

    li = list(LEXICONS).index("SocialSent")
    drows = []
    for dname in DOMAINS:
        X = subset.get(dname, np.zeros((len(targets), len(LEXICONS), 2)))
        for ti, t in enumerate(targets):
            k = X[ti, li]
            drows.append({"domain": dname, "target": t, "lexicon": "SocialSent", "sentiment_tokens": int(k.sum()),
                          "positivity": 100 * k[0] / k.sum() if k.sum() else np.nan,
                          "paper": PAPER["domain_positivity"].get((dname, t))})
    dom_df = pd.DataFrame(drows)

    srows = []
    for t in targets:
        st = int(pos_df[(pos_df.target == t) & (pos_df.lexicon == "SocialSent")]["sentiment_tokens"].iloc[0])
        srows.append({"target": t, "paragraphs": scale[f"{t} paragraphs"], "words_in_paragraphs": scale[f"{t} words"],
                      "paper_words_scaled_to_sample": PAPER["words_in_paragraphs"].get(t, np.nan) * n / 1024,
                      "socialsent_tokens": st,
                      "paper_socialsent_tokens_scaled_to_sample": PAPER["sentiment_tokens"].get(t, np.nan) * n / 1024})
    scale_df = pd.DataFrame(srows)

    trows = [{"target": t, "lexicon": l, "token": w, "count": c, "polarity": lex[l][w]}
             for (t, l), cnt in top.items() for w, c in cnt.most_common(25)]

    pos_df.round(2).to_csv(os.path.join(args.out_dir, "positivity.csv"), index=False)
    if has_pair:
        gap_df.round(3).to_csv(os.path.join(args.out_dir, "gaps.csv"), index=False)
        rob_df.round(3).to_csv(os.path.join(args.out_dir, "robustness.csv"), index=False)
    dom_df.round(2).to_csv(os.path.join(args.out_dir, "domains.csv"), index=False)
    scale_df.round(0).to_csv(os.path.join(args.out_dir, "scale.csv"), index=False)
    pd.DataFrame(trows).to_csv(os.path.join(args.out_dir, "top_tokens.csv"), index=False)

    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    rho = None
    if has_pair:
        fig_pair(pos_df[pos_df["target"].isin(PAIR)], gap_df, args.out_dir, args.dpi,
                 f"C4 paragraphs mentioning Jewish / Arab ({n} of 1,024 shards)", plt)
        fig_robustness(rob_df, args.out_dir, args.dpi, f"Robustness of the Jewish-Arab gap ({n} of 1,024 shards)", plt)
    if len(targets) > 2:
        fig_all_groups(pos_df, args.out_dir, args.dpi, f"Extension: C4 paragraphs per group ({n} of 1,024 shards)", plt)
        rho = fig_corpus_vs_table7(pos_df, args.out_dir, args.dpi, plt)

    sections = [("Positivity", pos_df)]
    if has_pair:
        sections.append(("Gap (Jewish - Arab)", gap_df))
        sections.append(("Robustness of the gap (each row changes one thing)", rob_df))
    sections += [("Domains (SocialSent)", dom_df), ("Scale vs the paper (paper numbers scaled by shards/1024)", scale_df)]
    with open(os.path.join(args.out_dir, "summary.md"), "w", encoding="utf-8") as f:
        f.write(f"# Arm B: C4 paragraph sentiment ({n} shards = {n}/1024 of C4.EN, {n_hosts} domains)\n\n")
        if rho is not None:
            f.write(f"Extension: Spearman between SocialSent corpus positivity and Table 7 positivity = {rho:.3f}\n\n")
        for name, d in sections:
            f.write(f"## {name}\n\n" + md_table(d.round(2)) + "\n")
    print(f"{n} shards, {n_hosts} domains, targets: {targets}")
    for name, d in sections:
        print(f"\n{name}\n" + d.round(2).to_string(index=False))
    if rho is not None:
        print(f"\nExtension: Spearman(corpus SocialSent positivity, Table 7) = {rho:.3f}")
    print(f"\nwrote {args.out_dir}")


if __name__ == "__main__":
    main()

## 3. Lexicons

SocialSent: the 250 subreddit lexicons released by Hamilton et al. (2016), here from the copy bundled in
`shifterator` (pinned commit). The `socialsent` Python package only contains the code that induces lexicons, not
these ready-made ones. Averaged over lexicons, words kept if present in at least 40, ethnic terms removed, top 250
on each side. VADER: words scoring above +1.0 or below −1.0. UnQover: our reconstruction (the paper's was not released).

In [ ]:
%cd /content/f4
!python scripts/build_lexicons.py

## 4. B1, the reproduction: Jewish and Arab on 12 seeded shards

About 10-30 min with 3 workers. Skips shards that are already done (B1 results ship with the repo, so this may finish at once).

In [ ]:
%cd /content/f4
!python scripts/scan_c4_paragraphs.py --n 12 --seed 0 --workers 3 --out_dir results/B1_c4_12_shards
!python scripts/count_sentiment.py --paragraphs results/B1_c4_12_shards --out_dir results/B1_c4_12_shards/analysis

In [ ]:
import pandas as pd
from IPython.display import Image, Markdown, display
d = "/content/f4/results/B1_c4_12_shards/analysis"
display(Markdown("**Positivity per lexicon** (paper: SocialSent 73.2 / 65.7)"))
display(pd.read_csv(f"{d}/positivity.csv"))
display(Markdown("**Jewish minus Arab gap** (paper: 7.5 / 2.0 / 1.4)"))
display(pd.read_csv(f"{d}/gaps.csv"))
display(Markdown("**Counts vs the paper's totals scaled to 12/1024 of C4**"))
display(pd.read_csv(f"{d}/scale.csv"))
display(Markdown("**NYT / Al Jazeera** (too few tokens on a sample; the paper used the full corpus)"))
display(pd.read_csv(f"{d}/domains.csv"))
display(Image(f"{d}/fig_positivity.png", width=850)); display(Image(f"{d}/fig_gap.png", width=850))
display(Markdown("**Robustness: each row changes one thing from the paper's setting**"))
display(pd.read_csv(f"{d}/robustness.csv").round(2))
display(Image(f"{d}/fig_robustness.png", width=1100))

## 5. B3, extension (optional): all 15 Table 7 groups on the same 12 shards

Not in the paper. Each group is matched by its spelling variants (e.g. "Middle-Eastern" or "Middle Eastern").
Caveats the paper itself gives: "African" also matches "African American", and "Black" and "White" have very common
non-ethnic senses. Read these numbers with that in mind. About 30 min.

In [ ]:
%cd /content/f4
!python scripts/scan_c4_paragraphs.py --table7_groups --n 12 --seed 0 --workers 3 --out_dir results/B3_ext_all_groups_12_shards
!python scripts/count_sentiment.py --paragraphs results/B3_ext_all_groups_12_shards --out_dir results/B3_ext_all_groups_12_shards/analysis

In [ ]:
import pandas as pd
from IPython.display import Image, display
d = "/content/f4/results/B3_ext_all_groups_12_shards/analysis"
p = pd.read_csv(f"{d}/positivity.csv")
display(p[p.lexicon == "SocialSent"].sort_values("positivity", ascending=False))
display(Image(f"{d}/fig_positivity_all_groups.png", width=1100)); display(Image(f"{d}/fig_corpus_vs_table7.png", width=750))

## 6. B4 (optional, free tier, several sessions): a quarter of C4.EN

The paper's NYT / Al Jazeera numbers came from 1.6k-7.9k sentiment tokens on the full corpus; 12 shards give
30-101. 256 shards (1/4 of C4.EN) give roughly 400-2,000, enough to compare. The full 1,024 shards are possible
the same way but take about four times as long.

The output goes to **Google Drive**, so a session that times out loses nothing: re-run this cell in a new session
and it continues. Free Colab stops idle sessions, so keep the tab open. The first shards print their time; multiply
by 256 / workers for the total, and split it over sessions as needed. The paragraphs take roughly 100 MB on Drive.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/f4_B4_c4_256_shards"      # one folder, reused by every session
%cd /content/f4
!python scripts/scan_c4_paragraphs.py --range 0 256 --workers 4 --cache_dir /content/hf_cache --out_dir "$OUT"

In [ ]:
# run when the scan reports 0 shards to scan (also works on a partial scan, with fewer shards)
%cd /content/f4
!python scripts/count_sentiment.py --paragraphs "$OUT" --out_dir "$OUT/analysis"
import pandas as pd
from IPython.display import Image, display
display(pd.read_csv(f"{OUT}/analysis/domains.csv")); display(pd.read_csv(f"{OUT}/analysis/gaps.csv"))
display(Image(f"{OUT}/analysis/fig_robustness.png", width=1100))

## 7. Download the results (or copy them to Google Drive)

In [ ]:
%cd /content/f4
!zip -qr /content/f4_arm_b_results.zip results/B* data/lexicons/MANIFEST.json
from google.colab import files
files.download("/content/f4_arm_b_results.zip")
# Drive instead: from google.colab import drive; drive.mount("/content/drive"); !cp -r results /content/drive/MyDrive/f4_results